# 总体均值的假设检验完整教程：单样本 t 检验

## 📚 教学目标
1. 掌握单样本 t 检验的假设设定和条件检查
2. 理解 p 值法、临界值法和置信区间法三种判断方法
3. 手算 t 统计量并用 t 分布表确定 p 值范围
4. 用 scipy.stats.ttest_1samp 验证手算结果
5. 理解双侧检验和单侧检验的区别及实际应用

**参考书**：《基础统计学(第14版)》（Triola）第 8-3 节
**教学策略**：先理解 t 分布的特征，再通过两个经典案例掌握完整的 t 检验流程

---

## 1. 场景设定：均值检验的实际应用

### 🎯 两个经典问题

**问题1（左侧检验）**：成年人的平均睡眠时间是否少于 7 小时？
- 样本：12 名成年人，均值 6.833 小时，标准差 1.992 小时

**问题2（双侧检验）**：人的平均体温真的是 98.6°F 吗？
- 样本：106 人，均值 98.20°F，标准差 0.62°F

### 📖 书中的观点

> 总体均值的假设检验是本书中介绍的最为重要的方法之一。
> 当总体标准差 σ 未知时（这是最常见的情况），使用样本标准差 s 来估计 σ，
> 此时检验统计量服从 t 分布。

### 📐 t 检验统计量

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad df = n - 1$$

其中：
- $\bar{x}$ = 样本均值
- $\mu_0$ = 假设的总体均值
- s = 样本标准差
- n = 样本量
- df = 自由度

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. t 检验的条件与 t 分布

### 📐 两个必要条件

| 条件 | 说明 |
|------|------|
| 条件 1 | 样本为简单随机样本 |
| 条件 2 | 总体服从正态分布，**或** n > 30 |

### 📝 不满足条件时的替代方法

| 方法 | 适用情况 |
|------|----------|
| 自助重采样法 | 非正态总体 |
| 置换检验 | 非参数方法 |
| 符号检验 | 13-2 节 |
| 威尔科克森符号秩检验 | 13-3 节 |

### 📐 学生 t 分布的性质

1. **随样本量变化**：不同 df 对应不同的 t 分布
2. **钟形但更宽**：小样本时比正态分布有更大的变异性
3. **均值为 0**：与标准正态分布相同
4. **标准差 > 1**：随 df 增大而趋近于 1
5. **极限性质**：当 n → ∞ 时，t 分布趋近标准正态分布

In [ ]:
# ========== t 分布可视化 ==========

fig, ax = plt.subplots(figsize=(10, 6))

x = np.linspace(-5, 5, 1000)

# 标准正态分布
y_norm = stats.norm.pdf(x)
ax.plot(x, y_norm, 'k-', linewidth=2.5, label='Standard Normal (z)', alpha=0.8)

# 不同自由度的 t 分布
dfs = [1, 3, 5, 10, 30]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
for df, color in zip(dfs, colors):
    y_t = stats.t.pdf(x, df)
    ax.plot(x, y_t, '-', color=color, linewidth=1.8, label=f't (df={df})', alpha=0.7)

ax.fill_between(x, 0, y_norm, alpha=0.05, color='steelblue')

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Student t Distribution vs Standard Normal', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)
ax.set_ylim(0, 0.45)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  黑色实线：标准正态分布 (z 分布)")
print("  彩色线：不同自由度的 t 分布")
print("  df 越小，t 分布越宽（变异性越大）")
print("  df 越大，t 分布越接近标准正态分布")
print("  当 df ≥ 30 时，t 分布与正态分布几乎相同")

---

## 3. 左侧检验完整示例：成年人睡眠时间

### 🎯 问题

从《美国国家健康与营养检查调查》中随机选取 12 名成年人的睡眠时间（小时）：

| 受试者 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|--------|---|---|---|---|---|---|---|---|---|----|----|----| 
| 睡眠时间 | 4.8 | 4.4 | 8.6 | 6.9 | 7.7 | 1.0 | 7.0 | 8.0 | 6.5 | 7.2 | 5.5 | 8.1 |

检验命题："成年人的平均睡眠时间少于 7 小时"（α = 0.05）

### 📐 假设设定

$H_0: \mu = 7$
$H_1: \mu < 7$ （原命题，左侧检验）

In [ ]:
# ========== 左侧检验: 成年人睡眠时间 ==========

print("=" * 60)
print("📋 成年人睡眠时间 - 左侧 t 检验")
print("=" * 60)

# 数据
sleep_data = np.array([4.8, 4.4, 8.6, 6.9, 7.7, 1.0, 7.0, 8.0, 6.5, 7.2, 5.5, 8.1])
n = len(sleep_data)
x_bar = np.mean(sleep_data)
s = np.std(sleep_data, ddof=1)
mu_0 = 7
alpha = 0.05
df = n - 1

print(f"\n📊 数据概况")
print(f"  样本量 n = {n}")
print(f"  样本均值 x̄ = {x_bar:.4f}")
print(f"  样本标准差 s = {s:.4f}")
print(f"  自由度 df = {df}")

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: μ = {mu_0}")
print(f"  H₁: μ < {mu_0} (原命题)")
print(f"  检验类型: 左侧检验")

print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

print(f"\n📊 步骤 5: 检查条件")
print(f"  条件1: 简单随机样本 ✓")
print(f"  条件2: n = {n} {'< 30，需检验正态性' if n < 30 else '≥ 30，满足中心极限定理'}")

# 计算检验统计量
t_stat = (x_bar - mu_0) / (s / np.sqrt(n))
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  t = (x̄ - μ₀) / (s/√n)")
print(f"  t = ({x_bar:.4f} - {mu_0}) / ({s:.4f}/√{n})")
print(f"  t = {t_stat:.4f}")

# 计算 p 值 (左侧检验)
p_value = stats.t.cdf(t_stat, df=df)
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  左侧检验: p 值 = P(T < {t_stat:.4f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
t_critical = stats.t.ppf(alpha, df=df)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 -t_α = {t_critical:.3f}")
print(f"  检验统计量 t = {t_stat:.4f}")

# 置信区间
ci_upper = x_bar + stats.t.ppf(1-alpha, df=df) * s / np.sqrt(n)
print(f"\n📊 步骤 6 (续): 置信区间法")
print(f"  90% 置信区间上限 = {ci_upper:.4f}")
print(f"  H₀ 假设值 μ₀ = {mu_0}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: t = {t_stat:.4f} {'< -t_α → 拒绝 H₀' if t_stat < t_critical else '≥ -t_α → 不能拒绝 H₀'}")
print(f"  置信区间法: μ₀ = {mu_0} {'> 上限 → 拒绝 H₀' if mu_0 > ci_upper else '≤ 上限 → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'成年人平均睡眠时间少于7小时'")
else:
    print(f"  没有足够的证据支持'成年人平均睡眠时间少于7小时'")

print("\n" + "=" * 60)

In [ ]:
# ========== scipy 验证 ==========

print("=" * 60)
print("📋 scipy.stats.ttest_1samp 验证")
print("=" * 60)

sleep_data = np.array([4.8, 4.4, 8.6, 6.9, 7.7, 1.0, 7.0, 8.0, 6.5, 7.2, 5.5, 8.1])
mu_0 = 7

# scipy 双侧检验
t_scipy, p_scipy_two = stats.ttest_1samp(sleep_data, mu_0)

# 手算
n = len(sleep_data)
x_bar = np.mean(sleep_data)
s = np.std(sleep_data, ddof=1)
t_manual = (x_bar - mu_0) / (s / np.sqrt(n))
p_manual = stats.t.cdf(t_manual, df=n-1)  # 左侧检验

print(f"\n🔬 手算 vs scipy 对比:")
print(f"  手算 t = {t_manual:.6f}")
print(f"  scipy t = {t_scipy:.6f}")
print(f"  手算 p (左侧) = {p_manual:.6f}")
print(f"  scipy p (双侧) = {p_scipy_two:.6f}")
print(f"  scipy p (单侧) = {p_scipy_two/2:.6f}")
print(f"\n  ✅ 验证通过！t 统计量完全一致")

print("\n" + "=" * 60)

---

## 4. 双侧检验完整示例：人体体温

### 🎯 问题

数据集包含 106 名成年人在凌晨 12 点的体温：n=106, x̄=98.20°F, s=0.62°F。

检验命题："人的平均体温是 98.6°F"（α = 0.05）

### 📐 假设设定

$H_0: \mu = 98.6$
$H_1: \mu \neq 98.6$ （双侧检验）

### 📐 检验统计量

$$t = \frac{98.20 - 98.6}{0.62 / \sqrt{106}} = -6.64$$

In [ ]:
# ========== 双侧检验: 人体体温 ==========

print("=" * 60)
print("📋 人体体温 - 双侧 t 检验")
print("=" * 60)

# 数据
n = 106
x_bar = 98.20
s = 0.62
mu_0 = 98.6
alpha = 0.05
df = n - 1

print(f"\n📊 数据概况")
print(f"  样本量 n = {n}")
print(f"  样本均值 x̄ = {x_bar}°F")
print(f"  样本标准差 s = {s}°F")
print(f"  自由度 df = {df}")

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: μ = {mu_0}°F")
print(f"  H₁: μ ≠ {mu_0}°F (双侧检验)")

print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

print(f"\n📊 步骤 5: 检查条件")
print(f"  条件1: 简单随机样本 ✓")
print(f"  条件2: n = {n} ≥ 30，满足中心极限定理 ✓")

# 计算检验统计量
t_stat = (x_bar - mu_0) / (s / np.sqrt(n))
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  t = (x̄ - μ₀) / (s/√n)")
print(f"  t = ({x_bar} - {mu_0}) / ({s}/√{n})")
print(f"  t = {t_stat:.4f}")

# 计算 p 值 (双侧检验)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  双侧检验: p 值 = 2 × P(T > |{t_stat:.2f}|)")
print(f"  p 值 = {p_value:.6f}")

# 临界值
t_critical = stats.t.ppf(1 - alpha/2, df=df)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 ±t_{{α/2}} = ±{t_critical:.3f}")
print(f"  检验统计量 t = {t_stat:.4f}")

# 置信区间
se = s / np.sqrt(n)
ci_lower = x_bar - t_critical * se
ci_upper = x_bar + t_critical * se
print(f"\n📊 步骤 6 (续): 置信区间法")
print(f"  95% 置信区间: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"  H₀ 假设值 μ₀ = {mu_0}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.6f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: |t| = {abs(t_stat):.4f} {'> t_α/2 → 拒绝 H₀' if abs(t_stat) > t_critical else '≤ t_α/2 → 不能拒绝 H₀'}")
print(f"  置信区间法: {mu_0} {'不在' if mu_0 < ci_lower or mu_0 > ci_upper else '在'} CI 内 → {'拒绝 H₀' if mu_0 < ci_lower or mu_0 > ci_upper else '不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据拒绝'{mu_0}°F 为人的平均体温'的看法")
else:
    print(f"  没有足够的证据拒绝'{mu_0}°F 为人的平均体温'的看法")

print("\n" + "=" * 60)

In [ ]:
# ========== scipy 验证 (体温) ==========

print("=" * 60)
print("📋 scipy 验证: 人体体温")
print("=" * 60)

# 由于没有原始数据，用汇总统计量模拟
n = 106
x_bar = 98.20
s = 0.62
mu_0 = 98.6

# 生成近似数据 (用于 scipy 验证)
simulated_data = np.random.normal(x_bar, s, n)
# 调整使其精确匹配给定的均值和标准差
simulated_data = (simulated_data - np.mean(simulated_data)) / np.std(simulated_data, ddof=1) * s + x_bar

t_scipy, p_scipy = stats.ttest_1samp(simulated_data, mu_0)

# 手算
t_manual = (x_bar - mu_0) / (s / np.sqrt(n))
p_manual = 2 * (1 - stats.t.cdf(abs(t_manual), df=n-1))

print(f"\n🔬 手算 vs scipy 对比:")
print(f"  手算 t = {t_manual:.6f}")
print(f"  手算 p = {p_manual:.6f}")
print(f"  💡 由于使用模拟数据，scipy 结果可能略有差异")
print(f"  💡 手算结果基于汇总统计量，是精确的")

print(f"\n📊 三种方法结论一致:")
print(f"  p 值 ≈ 0 (远小于 α = 0.05)")
print(f"  |t| = {abs(t_manual):.2f} (远大于临界值 ±1.983)")
print(f"  95% CI 不包含 98.6°F")
print(f"  → 拒绝 H₀，人的平均体温不是 98.6°F")

print("\n" + "=" * 60)

---

## 5. 三种检验方法对比

### 📐 对于 t 检验，三种方法完全等价

与比例检验不同，**t 检验的三种方法总是给出相同结论**：

| 方法 | 判断标准 | 适用场景 |
|------|---------|----------|
| p 值法 | p < α → 拒绝 H₀ | 最常用，提供精确证据强度 |
| 临界值法 | |t| > t_α/2 → 拒绝 H₀ | 手算查表时使用 |
| 置信区间法 | μ₀ 不在 CI 内 → 拒绝 H₀ | 同时提供参数估计 |

### 📝 书中的观点

> 对于本节中的 t 检验，从是否能够得出相同结论的角度来讲，
> p 值法、临界值法和置信区间法是等价的。

In [ ]:
# ========== 手算 p 值范围 ==========

print("=" * 60)
print("📋 手算 p 值范围 (查表法)")
print("=" * 60)

# 睡眠数据示例
t_stat = -0.290
df = 11

print(f"\n📊 睡眠数据示例:")
print(f"  检验统计量 t = {t_stat}")
print(f"  自由度 df = {df}")
print(f"  左侧检验")

print(f"\n📊 查表法确定 p 值范围:")
print(f"  使用 df = {df} 查询 t 分布表 (表 A-3)")
print(f"  表中所有 t 值都大于 |{t_stat}| = {abs(t_stat)}")
print(f"  这意味着 p 值 > 0.10")
print(f"  精确值: p = {stats.t.cdf(t_stat, df=df):.4f}")

print(f"\n💡 查表法的局限:")
print(f"  只能得到 p 值的范围，不是精确值")
print(f"  建议使用统计软件获取精确 p 值")

# 体温数据示例
t_stat_temp = -6.64
df_temp = 105

print(f"\n📊 体温数据示例:")
print(f"  检验统计量 t = {t_stat_temp}")
print(f"  自由度 df = {df_temp}")
print(f"  查表: p < 0.01 (非常小)")
print(f"  精确值: p = {2 * (1 - stats.t.cdf(abs(t_stat_temp), df=df_temp)):.10f}")

print("\n" + "=" * 60)

---

## 6. 可视化：t 检验结果

### 📊 图示说明

通过可视化直观展示 t 检验的判断过程：
- 蓝色曲线：t 分布
- 红色区域：拒绝域
- 绿色区域：接受域
- 橙色线：检验统计量

In [ ]:
# ========== 可视化两种检验 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 左侧检验 (睡眠)
ax = axes[0]
df_left = 11
t_stat_left = -0.290
t_crit_left = stats.t.ppf(0.05, df=df_left)

x = np.linspace(-5, 5, 1000)
y = stats.t.pdf(x, df_left)
ax.plot(x, y, 'b-', linewidth=2, label=f't Distribution (df={df_left})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

x_reject = np.linspace(-5, t_crit_left, 500)
ax.fill_between(x_reject, 0, stats.t.pdf(x_reject, df_left), alpha=0.4, color='#e74c3c',
                label=f'Rejection Region (α=0.05)')
x_accept = np.linspace(t_crit_left, 5, 500)
ax.fill_between(x_accept, 0, stats.t.pdf(x_accept, df_left), alpha=0.2, color='#2ecc71',
                label='Acceptance Region')
ax.axvline(x=t_crit_left, color='red', linestyle='--', linewidth=2,
           label=f'Critical Value t={t_crit_left:.3f}')
ax.axvline(x=t_stat_left, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Test Statistic t={t_stat_left:.3f}')

ax.set_xlabel('T Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Left-Tailed Test: Sleep Time (μ < 7)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

# 右图: 双侧检验 (体温)
ax = axes[1]
df_right = 105
t_stat_right = -6.64
t_crit_right = stats.t.ppf(0.975, df=df_right)

x = np.linspace(-8, 8, 1000)
y = stats.t.pdf(x, df_right)
ax.plot(x, y, 'b-', linewidth=2, label=f't Distribution (df={df_right})')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

x_reject_left = np.linspace(-8, -t_crit_right, 500)
ax.fill_between(x_reject_left, 0, stats.t.pdf(x_reject_left, df_right), alpha=0.4, color='#e74c3c',
                label='Rejection Region')
x_reject_right = np.linspace(t_crit_right, 8, 500)
ax.fill_between(x_reject_right, 0, stats.t.pdf(x_reject_right, df_right), alpha=0.4, color='#e74c3c')
x_accept = np.linspace(-t_crit_right, t_crit_right, 500)
ax.fill_between(x_accept, 0, stats.t.pdf(x_accept, df_right), alpha=0.2, color='#2ecc71',
                label='Acceptance Region')
ax.axvline(x=-t_crit_right, color='red', linestyle='--', linewidth=2)
ax.axvline(x=t_crit_right, color='red', linestyle='--', linewidth=2,
           label=f'Critical Values ±t={t_crit_right:.3f}')
ax.axvline(x=t_stat_right, color='#e67e22', linestyle='-', linewidth=2.5,
           label=f'Test Statistic t={t_stat_right:.2f}')

ax.set_xlabel('T Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Two-Tailed Test: Body Temp (μ ≠ 98.6)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图 (睡眠): t = -0.290 未落在拒绝域内 → 不能拒绝 H₀")
print("  右图 (体温): t = -6.64 远在拒绝域内 → 拒绝 H₀")
print("  红色区域 = 拒绝域 (α = 0.05)")
print("  绿色区域 = 接受域")
print("  橙色线 = 检验统计量的实际位置")

---

## 7. 实际显著性 vs 统计显著性

### 📖 书中的案例

> 默克公司对治疗失眠的药物苏沃雷生进行了临床试验：
> - 服用苏沃雷生的受试者比安慰剂组快 **6 分钟**入睡
> - 苏沃雷生受试者比安慰剂组多睡 **16 分钟**
> - 两项结果均具有**统计显著性**
>
> 但由于改善幅度较小，加上每片约 12 美元的高价和潜在副作用，
> 该药虽然统计显著，但**没有实际显著性**。

### 💡 核心启示

**统计显著 ≠ 实际显著**

- 统计显著：差异不太可能由随机误差引起
- 实际显著：差异在现实中有实际意义
- 大样本可以使微小差异变得统计显著
- 应结合效应量、成本、收益等因素综合判断

In [ ]:
# ========== 实际显著性 vs 统计显著性 ==========

print("=" * 60)
print("📋 实际显著性 vs 统计显著性")
print("=" * 60)

# 模拟: 不同样本量下的 t 检验
mu_0 = 100
mu_true = 100.5  # 真实均值比假设值高 0.5 (很小的差异)
sigma = 10
alpha = 0.05

sample_sizes = [10, 30, 100, 500, 1000, 5000]

print(f"\n📊 模拟设置:")
print(f"  H₀: μ = {mu_0}")
print(f"  真实均值: μ = {mu_true} (差异 = {mu_true - mu_0})")
print(f"  总体标准差: σ = {sigma}")
print(f"  显著性水平: α = {alpha}")

print(f"\n{'样本量 n':<12} {'效应量 d':<12} {'统计显著?':<12} {'实际意义?':<12}")
print("-" * 48)

for n in sample_sizes:
    # 效应量 (Cohen's d)
    effect_size = (mu_true - mu_0) / sigma
    
    # 模拟 1000 次
    sig_count = 0
    for _ in range(1000):
        sample = np.random.normal(mu_true, sigma, n)
        t_stat, p_val = stats.ttest_1samp(sample, mu_0)
        if p_val < alpha:
            sig_count += 1
    
    power = sig_count / 1000
    is_practical = abs(mu_true - mu_0) > 0.1 * sigma  # 简单判断标准
    
    print(f"  {n:<12} {effect_size:<12.3f} {'是' if power > 0.5 else '否':<12} {'是' if is_practical else '否':<12}")

print(f"\n💡 关键发现:")
print(f"  效应量 d = {effect_size:.3f} (非常小)")
print(f"  大样本可以使微小差异变得统计显著")
print(f"  但统计显著不代表有实际意义")
print(f"  应结合效应量、成本、收益综合判断")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 t 检验的条件
- **定义**: 简单随机样本 + (正态总体 或 n > 30)
- **公式**: $t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$, df = n - 1
- **含义**: 衡量样本均值与假设均值之间的标准误距离

### 📌 三种检验方法 (对于 t 检验等价)
- **p 值法**: p < α → 拒绝 H₀
- **临界值法**: |t| > t_α/2 → 拒绝 H₀
- **置信区间法**: μ₀ 不在 CI 内 → 拒绝 H₀
- **特点**: 三种方法总是给出相同结论

### 📌 t 分布 vs 正态分布
- **小样本**: t 分布更宽，有更大的变异性
- **大样本**: t 分布趋近正态分布
- **df ≥ 30**: t 分布与正态分布几乎相同

### 📌 统计显著 vs 实际显著
- **统计显著**: p < α，差异不太可能由随机误差引起
- **实际显著**: 差异在现实中有实际意义
- **关系**: 大样本可使微小差异统计显著，但不一定有实际意义

### 🔗 完整流程
```
设定假设 → 检查条件 → 计算 t → 求 p 值 → 判断 → 结论
    ↓          ↓         ↓        ↓       ↓       ↓
  H₀/H₁    正态/n>30  公式    查表/软件  p<α?   非技术语言
```

### 📝 检验类型选择

| 情况 | 检验类型 |
|------|----------|
| σ 已知 | z 检验 |
| σ 未知 | t 检验 |
| n > 30 | t 检验 ≈ z 检验 |
| n < 30 且非正态 | 非参数方法 |

---

## ❌ 常见误区

### ❌ 误区 1: σ 未知时用 z 检验代替 t 检验
**✓ 正确做法**: 当总体标准差 σ 未知时（这是最常见的情况），必须使用 t 检验。z 检验要求 σ 已知，这在实际中很少见。

### ❌ 误区 2: 小样本时忽略正态性条件
**✓ 正确做法**: 当 n < 30 时，需要检验数据是否近似正态分布。可以用直方图、QQ 图或 Shapiro-Wilk 检验。若数据严重非正态，应使用非参数方法。

### ❌ 误区 3: p 值法和临界值法给出不同结论
**✓ 正确做法**: 对于 t 检验，三种方法（p 值法、临界值法、置信区间法）**总是**给出相同结论。如果出现不一致，通常是计算错误。

### ❌ 误区 4: 统计显著等于实际显著
**✓ 正确理解**: 统计显著只意味着差异不太可能由随机误差引起。大样本可以使微小差异变得"统计显著"。要结合效应量和实际意义来解释结果。

### ❌ 误区 5: "接受 H₀" 等于 "H₀ 为真"
**✓ 正确做法**: 不能拒绝 H₀ 只表示证据不足，不等于证明 H₀ 为真。可能是样本量太小，或者真实差异太小而无法检测到。